In [14]:
import torch
from torch import nn
from d2l import torch as d2l

In [15]:
# NiN Block Architecture

def nin_block(
    out_channels: int,
    kernel_size: int,
    stride: int,
    padding: int,
) -> nn.Sequential:
    
    return nn.Sequential(
        
        # Spatial feature 추출
        nn.LazyConv2d(
            out_channels=out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
        ),
        nn.ReLU(),
        
        # 각 pixel의 channel vector에
        # Linear Transformation 적용
        nn.LazyConv2d(
            out_channels=out_channels,
            kernel_size=1,
        ),
        nn.ReLU(),
        
        nn.LazyConv2d(
            out_channels=out_channels,
            kernel_size=1,
        ),
        nn.ReLU(),
    )

In [16]:
# Tracing NiN Block Tensor Shape

# [B, C, H, W]
X = torch.randn(
    1,
    3,
    8,
    8,
)

block = nin_block(
    out_channels=16,
    kernel_size=3,
    stride=1,
    padding=1,
)

with torch.no_grad():
    for layer in block:
        X = layer(X)

        print(
            f"{layer.__class__.__name__:<12}",
            "output shape:",
            tuple(X.shape),
        )

Conv2d       output shape: (1, 16, 8, 8)
ReLU         output shape: (1, 16, 8, 8)
Conv2d       output shape: (1, 16, 8, 8)
ReLU         output shape: (1, 16, 8, 8)
Conv2d       output shape: (1, 16, 8, 8)
ReLU         output shape: (1, 16, 8, 8)


In [17]:
# 1x1 Convolution과 Linear Transformation 비교

# [B, C, H, W]
X_small = torch.randn(
    1,
    3,
    2,
    2,
)

pointwise_conv = nn.Conv2d(
    in_channels=3,
    out_channels=2,
    kernel_size=1,
)

bias = pointwise_conv.bias

assert bias is not None


with torch.no_grad():
    
    # Pytorch: 1x1 Convolution Layer Forward (baseline)
    convolution_output = pointwise_conv(
        X_small
    )
    
    # 왼쪽 위 pixel의 input_channel vector
    pixel_vector = X_small[
        0,
        :,
        0,
        0,
    ] # Column Vector: [R, G, B].T


    # weight tensor: [out, in, H=1, W=1]
    # 1x1 Conv의 weight를 matrix로 변환 -> [out, in]
    weight_matrix = pointwise_conv.weight[
        :,
        :,
        0,
        0,
    ]
    
    # [out, in] @ [in]
    linear_output = (
        weight_matrix @ pixel_vector
        + bias
    )
    
    
print(
    "1x1 Conv output:",
    convolution_output[0, :, 0, 0],
)
print(
    "Linear output:",
    linear_output,
)
print(
    "Same result:",
    torch.allclose(
        convolution_output[0, :, 0, 0],
        linear_output,
    ),
)

1x1 Conv output: tensor([ 1.0112, -0.0768])
Linear output: tensor([ 1.0112, -0.0768])
Same result: True
